# AeroRUL -- Phase 3: Sliding-Window Feature Engineering

**Objective**: Transform validated engine trajectories into leakage-safe, fixed-size temporal features for classical supervised regression.

---

## 1. Phase 3 Objective

Phase 2 established RUL targets, leakage-safe splitting, and condition-aware normalization. Phase 3 generates:

1. **Multi-scale rolling statistics** (mean, std, min, max, range)
2. **Degradation slopes** (linear OLS trend per window)
3. **Trend strength** (R² of linear fit)
4. **Temporal deltas** (short-term differences)
5. **Relative changes** (percentage-style differences)

All features are **strictly causal**: feature at cycle t uses only cycles ≤ t.

## 2. Load Phase 2 Data

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import SENSOR_COLUMNS, FEATURE_CONFIG, METADATA_COLUMNS, TARGET_COLUMN
from src.data_loader import load_cmapss
from src.rul import generate_rul_targets
from src.splitting import split_by_engine
from src.features import generate_window_features, generate_multiscale_features, get_feature_columns, generate_feature_metadata
from src.feature_validation import validate_feature_quality, validate_target_integrity, validate_engine_separation, feature_target_correlations

pd.set_option('display.max_columns', 30)
%matplotlib inline

# Load FD001 for demonstration
data = load_cmapss('FD001')
df_rul = generate_rul_targets(data['train'])
train_df, val_df = split_by_engine(df_rul)
print(f'Train: {len(train_df)} rows, {train_df["unit_id"].nunique()} engines')
print(f'Val:   {len(val_df)} rows, {val_df["unit_id"].nunique()} engines')

## 3. Verify Train/Validation Separation

In [ ]:
train_engines = set(train_df['unit_id'].unique())
val_engines = set(val_df['unit_id'].unique())
overlap = train_engines & val_engines
print(f'Train engines: {len(train_engines)}')
print(f'Val engines:   {len(val_engines)}')
print(f'Overlap:       {overlap if overlap else "EMPTY SET"} -> {"PASS" if not overlap else "FAIL"}')

## 4. Feature Configuration

In [ ]:
print('Feature Configuration:')
for key, value in FEATURE_CONFIG.items():
    print(f'  {key}: {value}')

n_sensors = len(SENSOR_COLUMNS)
n_windows = len(FEATURE_CONFIG['windows'])
n_stats = len([s for s in FEATURE_CONFIG['statistics'] if s in {'mean','std','min','max','range','slope','r2'}])
n_deltas = len(FEATURE_CONFIG['deltas'])
n_relchanges = len(FEATURE_CONFIG['relative_changes'])
total = n_sensors * n_windows * n_stats + n_sensors * n_deltas + n_sensors * n_relchanges
print(f'\nExpected features: {n_sensors} sensors × {n_windows} windows × {n_stats} stats '
      f'+ {n_sensors}×{n_deltas} deltas + {n_sensors}×{n_relchanges} relchanges = {total}')

## 5. Single-Window Feature Generation

In [ ]:
# Generate features for a single window W=30
W = 30
feats_30 = generate_window_features(train_df, window_size=W, sensor_columns=SENSOR_COLUMNS[:5])
print(f'W={W}: {len(feats_30)} rows, {len(get_feature_columns(feats_30))} features')
print(f'First cycle: {feats_30["cycle"].min()} (expected: {W})')
feats_30.head()

## 6. Rolling Statistics Example

In [ ]:
# Visualize raw vs rolling mean for engine 1
eid = sorted(train_df['unit_id'].unique())[0]
raw = train_df[train_df['unit_id'] == eid].sort_values('cycle')
feat = feats_30[feats_30['unit_id'] == eid].sort_values('cycle')

sensor = 'sensor_2'
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(raw['cycle'], raw[sensor], linewidth=0.6, alpha=0.6, label='Raw')
axes[0].plot(feat['cycle'], feat[f'{sensor}_mean_30'], linewidth=1.5, label='Mean (W=30)')
axes[0].set_title(f'{sensor}: Raw vs Rolling Mean', fontweight='bold')
axes[0].set_xlabel('Cycle'); axes[0].legend()

axes[1].plot(feat['cycle'], feat[f'{sensor}_std_30'], linewidth=1.2, color='coral')
axes[1].set_title(f'{sensor}: Rolling Std (W=30)', fontweight='bold')
axes[1].set_xlabel('Cycle')
plt.tight_layout(); plt.show()

## 7. Degradation Slope Example

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(raw['cycle'], raw[sensor], linewidth=0.6, alpha=0.5, label='Raw')
ax2 = ax.twinx()
ax2.plot(feat['cycle'], feat[f'{sensor}_slope_30'], color='red', linewidth=1.2, label='Slope (W=30)')
ax.set_xlabel('Cycle'); ax.set_ylabel('Sensor Value')
ax2.set_ylabel('Slope', color='red')
ax.legend(loc='lower left'); ax2.legend(loc='upper left')
ax.set_title(f'{sensor}: Degradation Slope', fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Multi-Scale Feature Construction

In [ ]:
# Generate full multi-scale features
train_feats = generate_multiscale_features(train_df)
val_feats = generate_multiscale_features(val_df)

feat_cols = get_feature_columns(train_feats)
print(f'Total features: {len(feat_cols)}')
print(f'Train: {len(train_feats)} rows, Val: {len(val_feats)} rows')
print(f'Max window: {max(FEATURE_CONFIG["windows"])} -> first valid cycle')

## 9. Multi-Scale Slopes

In [ ]:
feat_e1 = train_feats[train_feats['unit_id'] == eid].sort_values('cycle')
fig, ax = plt.subplots(figsize=(14, 5))
for W in FEATURE_CONFIG['windows']:
    col = f'{sensor}_slope_{W}'
    if col in feat_e1.columns:
        ax.plot(feat_e1['cycle'], feat_e1[col], linewidth=1.0, label=f'W={W}')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title(f'Engine {eid}: Multi-Scale {sensor} Slopes', fontweight='bold')
ax.set_xlabel('Cycle'); ax.set_ylabel('Slope'); ax.legend()
plt.tight_layout(); plt.show()

## 10. Feature Quality Validation

In [ ]:
q = validate_feature_quality(train_feats)
print('Feature Quality (Training):')
print(f'  Features:          {q["n_features"]}')
print(f'  Rows:              {q["n_rows"]}')
print(f'  NaN columns:       {q["n_cols_with_nan"]}')
print(f'  Inf columns:       {q["n_cols_with_inf"]}')
print(f'  Constant features: {q["n_constant_features"]}')
print(f'  Duplicate rows:    {q["n_duplicate_rows"]}')

t = validate_target_integrity(train_feats, train_df)
print(f'\nTarget Integrity: all_valid={t["all_valid"]}')

es = validate_engine_separation(train_feats, val_feats)
print(f'Engine Separation: no_overlap={es["no_overlap"]}')

## 11. Feature-Target Analysis

In [ ]:
# Top features by Pearson correlation with RUL
corrs = feature_target_correlations(train_feats, method='pearson', top_n=20)
print('Top 20 features by |Pearson correlation| with RUL:')
corrs.head(20)

In [ ]:
# Feature-target scatter for top 4
top4 = corrs.head(4)['feature'].tolist()
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
sample = train_feats.sample(n=min(2000, len(train_feats)), random_state=42)
for i, feat in enumerate(top4):
    axes[i].scatter(sample[feat], sample['rul'], alpha=0.2, s=5)
    r = train_feats[[feat, 'rul']].dropna().corr().iloc[0,1]
    axes[i].set_title(f'r={r:.3f}', fontweight='bold')
    axes[i].set_xlabel(feat, fontsize=8)
    axes[i].set_ylabel('RUL' if i==0 else '')
plt.suptitle('Top Feature-Target Relationships', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 12. Phase 3 Findings

### Summary

1. **Causal feature construction**: All features at cycle t use only cycles ≤ t. Verified by leakage test.
2. **Multi-scale representation**: Windows of 10, 20, 30, 50, 75 cycles capture short/medium/long-term degradation.
3. **Feature quality**: Checked for NaN, Inf, constant features, duplicates.
4. **Engine separation preserved**: Zero overlap between train and validation engine IDs.
5. **Phase 2 normalization consumed**: FD002/FD004 use normalized inputs.

### Methodological Note
The optimal window size and feature selection will be determined in Phase 4 through model validation performance.

### Next Phase
Phase 4: Classical Regression Model Benchmarking using the leakage-safe feature datasets.